In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import sklearn
import cv2

print(f"TensorFlow version: {tf.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print("Environnement prêt !")

from sklearn.model_selection import train_test_split
from PIL import Image

In [ ]:
IMAGE_SIZE = (32, 32)
data, labels = [], []

Rendre toutes les images de la même taille (obligatoire pour le CNN), /255.0 normalise les pixels entre 0 et 1 pour accélérer l'apprentissage.

In [ ]:
for class_id in range(43):  # 43 classes
    folder = f"dataset/{class_id}/"
    for img_file in os.listdir(folder):
        img = Image.open(folder + img_file).resize(IMAGE_SIZE)
        data.append(np.array(img))
        labels.append(class_id)

In [ ]:
data = np.array(data) / 255.0        # Normalisation [0, 1]
labels = np.array(labels)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42
)

# 3. Construire le CNN

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [ ]:
y_train_cat = to_categorical(y_train, 43)
y_test_cat  = to_categorical(y_test, 43)

Explication de chaque couche :

Conv2D — détecter des traits locaux (bords, formes, couleurs)
MaxPooling2D — réduire la taille de l'image tout en gardant l'essentiel
Flatten — transformer la grille 2D en un vecteur 1D
Dense — couches entièrement connectées qui apprennent les associations
Dropout(0.5) — désactiver aléatoirement 50% des neurones à chaque passage pour éviter le surapprentissage
Softmax — convertit la sortie en probabilités pour chaque classe

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),             # Évite le surapprentissage
    Dense(43, activation='softmax')   # 43 classes en sortie
])

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

# 4. Entrainement du modèle

history = model.fit(
    X_train, y_train_cat,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)

## 4.1 Visualisation des courbes d'aprentissage

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Précision'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Perte'); plt.legend()

plt.show()

## 5. Evaluation du modèle

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [ ]:
# Évaluation globale
loss, acc = model.evaluate(X_test, y_test_cat)
print(f"Précision sur le jeu de test : {acc:.2%}")

In [ ]:
# Matrice de confusion
y_pred = model.predict(X_test).argmax(axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.title('Matrice de confusion')
plt.show()

## 5.1 Test sur une nouvelle image

In [ ]:
# Tester sur une nouvelle image
img = Image.open("nouvelle_image.jpg").resize((32, 32))
img_array = np.array(img) / 255.0
img_array = img_array.reshape(1, 32, 32, 3)
prediction = model.predict(img_array).argmax()
print(f"Panneau détecté : classe {prediction}")

## 6. Interface graphique avec Tkinter

In [2]:
import tkinter as tk
from tkinter import filedialog
from PIL import ImageTk

def predict_image():
    path = filedialog.askopenfilename()
    img = Image.open(path).resize((32, 32))
    arr = np.array(img) / 255.0
    pred = model.predict(arr.reshape(1,32,32,3)).argmax()
    label_result.config(text=f"Panneau : classe {pred}")
    photo = ImageTk.PhotoImage(Image.open(path).resize((200, 200)))
    panel.config(image=photo); panel.image = photo

root = tk.Tk(); root.title("Détection de panneaux")
tk.Button(root, text="Choisir une image", command=predict_image).pack()
panel = tk.Label(root); panel.pack()
label_result = tk.Label(root, font=("Arial", 14)); label_result.pack()
root.mainloop()

ModuleNotFoundError: No module named 'tkinter'